# ASOS 일자료 기반 내일 평균기온 예측

관측일 `t`의 확정 일자료로 `t+2`일 평균기온을 예측합니다. 일자료가 다음 날 제공되므로 실제 서비스에서는 **어제 자료로 내일을 예측**합니다.

API 키는 코드에 직접 입력하지 않습니다. Colab 왼쪽의 열쇠 모양 **Secrets** 메뉴에서 `KMA_API_KEY`를 등록하고 이 노트북의 접근 권한을 켜주세요. 공공데이터포털에서 제공하는 일반 인증키(Decoding)를 권장합니다.

In [ ]:
!pip -q install 'catboost>=1.2,<2' 'scikit-learn>=1.3,<2' 'joblib>=1.3,<2'

In [ ]:
from __future__ import annotations

import json
import os
import time
from dataclasses import dataclass, asdict
from datetime import date, datetime, timedelta
from pathlib import Path
from urllib.parse import unquote

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 1. 실행 설정과 API 키 로드

In [ ]:
@dataclass(frozen=True)
class Config:
    station_id: str = '108'            # 서울
    start_date: str = '20000101'
    end_date: str = (date.today() - timedelta(days=1)).strftime('%Y%m%d')
    train_end: str = '2018-12-31'
    valid_end: str = '2022-12-31'
    lower_quantile: float = 0.005
    upper_quantile: float = 0.995
    api_url: str = 'https://apis.data.go.kr/1360000/AsosDalyInfoService/getWthrDataList'
    artifact_dir: str = '/content/weather_artifacts'

CFG = Config()

def load_api_key() -> str:
    key = os.getenv('KMA_API_KEY', '').strip()
    try:
        from google.colab import userdata
        key = (userdata.get('KMA_API_KEY') or key).strip()
    except (ImportError, KeyError, RuntimeError):
        pass
    if not key:
        raise RuntimeError('Colab Secrets에 KMA_API_KEY를 등록해 주세요.')
    return unquote(key)

KMA_API_KEY = load_api_key()
print('설정 완료:', CFG.station_id, CFG.start_date, '~', CFG.end_date)


## 2. 기상청 API 필드와 기존 44개 입력 변수 정의

In [ ]:
API_TO_KOREAN = {
    'tm': '일시', 'avgTa': '평균기온(°C)', 'minTa': '최저기온(°C)', 'maxTa': '최고기온(°C)',
    'hr1MaxRn': '1시간 최다강수량(mm)', 'sumRn': '일강수량(mm)',
    'maxInsWs': '최대 순간 풍속(m/s)', 'maxInsWsWd': '최대 순간 풍속 풍향(16방위)',
    'maxWs': '최대 풍속(m/s)', 'maxWsWd': '최대 풍속 풍향(16방위)',
    'avgWs': '평균 풍속(m/s)', 'hr24SumRws': '풍정합(100m)', 'maxWd': '최다풍향(16방위)',
    'avgTd': '평균 이슬점온도(°C)', 'minRhm': '최소 상대습도(%)', 'avgRhm': '평균 상대습도(%)',
    'avgPv': '평균 증기압(hPa)', 'avgPa': '평균 현지기압(hPa)',
    'maxPs': '최고 해면기압(hPa)', 'minPs': '최저 해면기압(hPa)', 'avgPs': '평균 해면기압(hPa)',
    'ssDur': '가조시간(hr)', 'sumSsHr': '합계 일조시간(hr)',
    'hr1MaxIcsr': '1시간 최다일사량(MJ/m2)', 'sumGsr': '합계 일사량(MJ/m2)',
    'ddMefs': '일 최심신적설(cm)', 'ddMes': '일 최심적설(cm)', 'sumDpthFhsc': '합계 3시간 신적설(cm)',
    'avgTca': '평균 전운량(1/10)', 'avgLmac': '평균 중하층운량(1/10)',
    'avgTs': '평균 지면온도(°C)', 'minTg': '최저 초상온도(°C)',
    'avgCm5Te': '평균 5cm 지중온도(°C)', 'avgCm10Te': '평균 10cm 지중온도(°C)',
    'avgCm20Te': '평균 20cm 지중온도(°C)', 'avgCm30Te': '평균 30cm 지중온도(°C)',
    'avgM05Te': '0.5m 지중온도(°C)', 'avgM10Te': '1.0m 지중온도(°C)',
    'avgM15Te': '1.5m 지중온도(°C)', 'avgM30Te': '3.0m 지중온도(°C)',
    'avgM50Te': '5.0m 지중온도(°C)', 'sumLrgEv': '합계 대형증발량(mm)',
    'sumSmlEv': '합계 소형증발량(mm)', 'n99Rn': '9-9강수(mm)', 'sumFogDur': '안개 계속시간(hr)'
}

FEATURE_COLUMNS = [name for key, name in API_TO_KOREAN.items() if key != 'tm']
assert len(FEATURE_COLUMNS) == 44, len(FEATURE_COLUMNS)
TARGET_COLUMN = '목표_평균기온(°C)'
DATE_COLUMN = '일시'
print('입력 변수:', len(FEATURE_COLUMNS), '개')


## 3. ASOS 일자료 수집
연 단위로 요청하고 페이지를 순회하여 장기 수집 실패 가능성을 줄입니다.

In [ ]:
def request_page(start_dt: str, end_dt: str, page_no: int = 1, rows: int = 999) -> dict:
    params = {
        'serviceKey': KMA_API_KEY, 'pageNo': page_no, 'numOfRows': rows, 'dataType': 'JSON',
        'dataCd': 'ASOS', 'dateCd': 'DAY', 'startDt': start_dt, 'endDt': end_dt,
        'stnIds': CFG.station_id,
    }
    response = requests.get(CFG.api_url, params=params, timeout=30)
    response.raise_for_status()
    try:
        payload = response.json()
    except requests.JSONDecodeError as exc:
        raise RuntimeError(f'JSON이 아닌 응답입니다: {response.text[:300]}') from exc
    header = payload.get('response', {}).get('header', {})
    if header.get('resultCode') not in (None, '00', '0'):
        raise RuntimeError(f"API 오류: {header.get('resultCode')} {header.get('resultMsg')}")
    return payload.get('response', {}).get('body', {})

def fetch_asos_daily(start_date: str, end_date: str) -> pd.DataFrame:
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    records = []
    for year in range(start.year, end.year + 1):
        chunk_start = max(start, pd.Timestamp(year=year, month=1, day=1))
        chunk_end = min(end, pd.Timestamp(year=year, month=12, day=31))
        page = 1
        while True:
            body = request_page(chunk_start.strftime('%Y%m%d'), chunk_end.strftime('%Y%m%d'), page)
            items = body.get('items') or {}
            rows = items.get('item') or []
            if isinstance(rows, dict):
                rows = [rows]
            records.extend(rows)
            total = int(body.get('totalCount') or 0)
            if page * 999 >= total:
                break
            page += 1
        print(year, '완료:', len(records), '누적 행')
        time.sleep(0.1)
    if not records:
        raise RuntimeError('수집된 ASOS 일자료가 없습니다. API 키와 요청 기간을 확인하세요.')
    frame = pd.DataFrame(records).rename(columns=API_TO_KOREAN)
    missing = sorted(set([DATE_COLUMN, *FEATURE_COLUMNS]) - set(frame.columns))
    if missing:
        raise RuntimeError(f'API 응답에 필요한 필드가 없습니다: {missing}')
    frame = frame[[DATE_COLUMN, *FEATURE_COLUMNS]].copy()
    frame[DATE_COLUMN] = pd.to_datetime(frame[DATE_COLUMN])
    return frame.drop_duplicates(DATE_COLUMN).sort_values(DATE_COLUMN).reset_index(drop=True)

CACHE_PATH = Path(f'/content/asos_daily_{CFG.station_id}_{CFG.start_date}_{CFG.end_date}.csv')
if CACHE_PATH.exists():
    raw_df = pd.read_csv(CACHE_PATH, parse_dates=[DATE_COLUMN])
    print('캐시 사용:', CACHE_PATH)
else:
    raw_df = fetch_asos_daily(CFG.start_date, CFG.end_date)
    raw_df.to_csv(CACHE_PATH, index=False, encoding='utf-8-sig')
raw_df.info()


## 4. 목표값 생성과 시간 분할
행 단위 `shift` 대신 실제 달력의 `t+2일`과 결합하여 누락된 날짜가 있어도 목표가 어긋나지 않게 합니다.

In [ ]:
for column in FEATURE_COLUMNS:
    raw_df[column] = pd.to_numeric(raw_df[column], errors='coerce')

target_lookup = raw_df[[DATE_COLUMN, '평균기온(°C)']].copy()
target_lookup[DATE_COLUMN] = target_lookup[DATE_COLUMN] - pd.Timedelta(days=2)
target_lookup = target_lookup.rename(columns={'평균기온(°C)': TARGET_COLUMN})
dataset = raw_df.merge(target_lookup, on=DATE_COLUMN, how='left').dropna(subset=[TARGET_COLUMN])

train_df = dataset[dataset[DATE_COLUMN] <= CFG.train_end].copy()
valid_df = dataset[(dataset[DATE_COLUMN] > CFG.train_end) & (dataset[DATE_COLUMN] <= CFG.valid_end)].copy()
test_df = dataset[dataset[DATE_COLUMN] > CFG.valid_end].copy()
if min(len(train_df), len(valid_df), len(test_df)) == 0:
    raise RuntimeError('분할 중 비어 있는 구간이 있습니다. 수집 기간 또는 분할 날짜를 조정하세요.')
print({'train': len(train_df), 'valid': len(valid_df), 'test': len(test_df)})


## 5. 누수 없는 결측치·이상치 전처리
강수·적설·안개·증발량 계열 결측은 0으로 처리합니다. 나머지는 과거값 전파 후 학습 구간 중앙값으로 채웁니다. 이상치는 학습 구간의 0.5%~99.5% 분위 범위로 clipping하며 검증·테스트 기준은 따로 학습하지 않습니다.

In [ ]:
class WeatherPreprocessor:
    def __init__(self, feature_columns, lower_quantile=0.005, upper_quantile=0.995):
        self.feature_columns = list(feature_columns)
        self.lower_quantile = lower_quantile
        self.upper_quantile = upper_quantile
        self.zero_fill_columns = [
            c for c in self.feature_columns
            if any(token in c for token in ('강수', '적설', '안개', '증발량'))
        ]

    def _prepare(self, frame):
        data = frame[self.feature_columns].copy()
        for column in self.feature_columns:
            data[column] = pd.to_numeric(data[column], errors='coerce')
        data[self.zero_fill_columns] = data[self.zero_fill_columns].fillna(0.0)
        return data.ffill()

    def fit(self, frame):
        data = self._prepare(frame)
        self.medians_ = data.median()
        filled = data.fillna(self.medians_)
        self.lower_bounds_ = filled.quantile(self.lower_quantile)
        self.upper_bounds_ = filled.quantile(self.upper_quantile)
        return self

    def transform(self, frame):
        data = self._prepare(frame).fillna(self.medians_)
        data = data.clip(self.lower_bounds_, self.upper_bounds_, axis=1)
        if data.isna().any().any():
            raise ValueError('전처리 후 결측치가 남아 있습니다.')
        return data.astype('float32')

preprocessor = WeatherPreprocessor(FEATURE_COLUMNS, CFG.lower_quantile, CFG.upper_quantile)
X_train = preprocessor.fit(train_df).transform(train_df)
X_valid = preprocessor.transform(valid_df)
X_test = preprocessor.transform(test_df)
y_train = train_df[TARGET_COLUMN].astype('float32')
y_valid = valid_df[TARGET_COLUMN].astype('float32')
y_test = test_df[TARGET_COLUMN].astype('float32')
print('전처리 완료:', X_train.shape, X_valid.shape, X_test.shape)


## 6. Persistence 기준모델과 CatBoost 강화 모델 학습

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'rmse': float(mean_squared_error(y_true, y_pred) ** 0.5),
        'r2': float(r2_score(y_true, y_pred)),
    }

baseline_pred = X_test['평균기온(°C)'].to_numpy()
baseline_metrics = regression_metrics(y_test, baseline_pred)
print('Persistence baseline:', baseline_metrics)

model = CatBoostRegressor(
    loss_function='MAE', eval_metric='MAE', iterations=3000, learning_rate=0.025,
    depth=8, l2_leaf_reg=5.0, random_strength=0.5, bagging_temperature=0.5,
    random_seed=RANDOM_SEED, allow_writing_files=False, verbose=100,
)
model.fit(X_train, y_train, eval_set=(X_valid, y_valid), use_best_model=True, early_stopping_rounds=200)


## 7. 전체·계절별 평가

In [ ]:
valid_pred = model.predict(X_valid)
test_pred = model.predict(X_test)
valid_metrics = regression_metrics(y_valid, valid_pred)
test_metrics = regression_metrics(y_test, test_pred)
print('Validation:', valid_metrics)
print('Test:', test_metrics)
print('MAE improvement vs baseline:', baseline_metrics['mae'] - test_metrics['mae'])

result = test_df[[DATE_COLUMN]].copy()
result['target_date'] = result[DATE_COLUMN] + pd.Timedelta(days=2)
result['actual'] = y_test.to_numpy()
result['prediction'] = test_pred
result['absolute_error'] = np.abs(result['actual'] - result['prediction'])
result['season'] = result['target_date'].dt.month.map({12:'겨울',1:'겨울',2:'겨울',3:'봄',4:'봄',5:'봄',6:'여름',7:'여름',8:'여름',9:'가을',10:'가을',11:'가을'})
display(result.groupby('season', sort=False)['absolute_error'].agg(['count', 'mean', 'median', 'max']))

plt.figure(figsize=(16, 5))
plt.plot(result['target_date'], result['actual'], label='actual', linewidth=1)
plt.plot(result['target_date'], result['prediction'], label='prediction', linewidth=1)
plt.title('Daily average temperature: actual vs prediction')
plt.ylabel('°C')
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 8. FastAPI에서 사용할 산출물 저장

In [ ]:
artifact_dir = Path(CFG.artifact_dir)
artifact_dir.mkdir(parents=True, exist_ok=True)
model_version = f"catboost-asos-daily-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
model_path = artifact_dir / 'catboost_daily_temperature.cbm'
preprocessor_path = artifact_dir / 'preprocessor.joblib'
metadata_path = artifact_dir / 'model_metadata.json'

model.save_model(model_path)
joblib.dump(preprocessor, preprocessor_path)
(artifact_dir / 'feature_columns.json').write_text(json.dumps(FEATURE_COLUMNS, ensure_ascii=False, indent=2), encoding='utf-8')
metadata = {
    'model_version': model_version, 'model_type': 'CatBoostRegressor',
    'station_id': CFG.station_id, 'target': TARGET_COLUMN, 'forecast_offset_days': 2,
    'feature_count': len(FEATURE_COLUMNS), 'train_start': str(train_df[DATE_COLUMN].min().date()),
    'train_end': str(train_df[DATE_COLUMN].max().date()), 'valid_metrics': valid_metrics,
    'test_metrics': test_metrics, 'baseline_metrics': baseline_metrics, 'config': asdict(CFG),
}
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
result.to_csv(artifact_dir / 'test_predictions.csv', index=False, encoding='utf-8-sig')
print('저장 완료:', artifact_dir)
print('\n'.join(str(path) for path in sorted(artifact_dir.iterdir())))


## 9. 최신 일자료로 1회 예측 확인
전일 일자료가 아직 공개되지 않았거나 필수값이 누락되면 실행 시 오류가 발생할 수 있습니다. 운영 단계에서는 FastAPI 작업이 재시도와 상태 기록을 담당합니다.

In [ ]:
observation_date = pd.Timestamp(date.today() - timedelta(days=1))
latest = fetch_asos_daily(observation_date.strftime('%Y%m%d'), observation_date.strftime('%Y%m%d'))
latest_X = preprocessor.transform(latest)
latest_prediction = float(model.predict(latest_X)[0])
prediction_payload = {
    'station_id': CFG.station_id,
    'observation_date': str(observation_date.date()),
    'predicted_for_date': str((observation_date + pd.Timedelta(days=2)).date()),
    'predicted_avg_temperature': round(latest_prediction, 2),
    'model_version': model_version,
}
print(json.dumps(prediction_payload, ensure_ascii=False, indent=2))

# 필요하면 아래 주석을 해제하여 산출물을 한 번에 내려받습니다.
# !zip -qr /content/weather_artifacts.zip /content/weather_artifacts
# from google.colab import files
# files.download('/content/weather_artifacts.zip')
